# 🧪 EXERCISE: Longitudinal Behavioural Analysis

> **Instructions:** This notebook has intentional gaps — paths, loops, calculations and plot labels that you must complete. Look for `# TODO` and `???` markers. Run each cell in order once you have filled in the blanks.

## Learning objectives

- Set file paths for a single participant across multiple sessions
- Load and label PsychoPy CSV files using glob and a for loop
- Compute accuracy, trial counts and response-time statistics per session
- Run a paired t-test to detect within-participant learning effects

---

# VISION Task — Longitudinal Behavioural Analysis (One Patient, All Sessions)

Tracks accuracy, response time, and spatial performance across all sessions for a single patient.

**SCOPE: one participant × all sessions**

---
## 0 Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import glob
import re

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

# ── TODO 1: Set your paths ────────────────────────────────────────────────────
PARTICIPANT_ID   = '???'          # e.g. 'JF'
DATA_DIR         = Path('???')    # folder containing PsychoPy CSV files
GAZE_METRICS_DIR = Path('???')    # folder containing gaze metrics CSV files

MIN_ACCURACY      = 0.75
POSITION_STEP_DEG = 4.5
_REQUIRED_COLS = {'trial_num', 'axis_deg', 'position_num', 'side', 'is_correct'}


---
## 1 Load All Sessions

In [ ]:
def load_session_csv(path, session_label=None):
    """Load one PsychoPy VISION_task CSV; return cleaned DataFrame or None."""
    df = pd.read_csv(path, encoding='utf-8-sig')
    df = df.dropna(axis=1, how='all')
    missing = _REQUIRED_COLS - set(df.columns)
    if missing:
        print(f'  [skip] {Path(path).name} — missing: {missing}'); return None
    df = df.dropna(subset=['trial_num', 'is_correct']).reset_index(drop=True)
    df['trial_num']    = df['trial_num'].astype(int)
    df['is_correct']   = df['is_correct'].astype(int)
    df['position_num'] = df['position_num'].astype(int)
    df['axis_deg']     = df['axis_deg'].astype(int)
    if 'response_time_s' in df.columns and 'rt' not in df.columns:
        df['rt'] = pd.to_numeric(df['response_time_s'], errors='coerce')
    elif 'rt' not in df.columns:
        df['rt'] = np.nan
    df['axis_label']       = df['axis_deg'].map({0:'Horizontal',45:'Diagonal BL→TR',135:'Diagonal TL→BR'})
    df['eccentricity_deg'] = POSITION_STEP_DEG * df['position_num']
    df['hemifield']        = df['side'].map({'R':'Right','L':'Left'})
    if 'participant' not in df.columns:
        df['participant'] = Path(path).stem.split('_')[0]
    if session_label:
        df['session'] = session_label
    elif 'session' not in df.columns:
        df['session'] = 'S1'
    return df

# ── TODO 2: Find and load all session CSVs for this participant ───────────────
# Use glob.glob() to search DATA_DIR for files matching this participant.
# Exclude any files containing 'gaze_metrics' in the name.
# For each file found, call load_session_csv() with label 'S1', 'S2', etc.
# Store results in `sessions_df` (list of DataFrames).

all_csv = sorted([
    p for p in glob.glob(str(DATA_DIR / f'*{PARTICIPANT_ID}*VISION_task*.csv'))
    if 'gaze_metrics' not in Path(p).name
])

sessions_df = []
for i, path in enumerate(all_csv):
    # TODO: call load_session_csv and append non-None results to sessions_df
    pass   # ← replace this

if not sessions_df:
    print(f'No sessions found for {PARTICIPANT_ID} in {DATA_DIR}')
else:
    all_df = pd.concat(sessions_df, ignore_index=True)
    print(f'Loaded {len(sessions_df)} sessions, {len(all_df)} trials total')


---
## 2 Session-Level Summary Table

In [ ]:
# ── TODO 3: Build the session summary table ───────────────────────────────────
# Loop over `sessions_df`. For each session DataFrame compute:
#   - session label (s_df['session'].iloc[0])
#   - n: total trials
#   - n_c: correct trials
#   - accuracy: n_c / n
#   - mean and SD of RT (response time) for correct trials only
#   - whether accuracy >= MIN_ACCURACY (pass/fail)
# Append a row dict to `rows` and print the table.

rows = []
for s_df in sessions_df:
    sess = s_df['session'].iloc[0]
    # Your code here:
    n   = ???
    n_c = ???
    rt  = s_df[s_df['is_correct'] == 1]['rt'].dropna()
    rows.append({
        'Session'     : sess,
        'Trials'      : n,
        'Correct'     : n_c,
        'Accuracy'    : f'{n_c/n:.1%}',
        'Pass ≥75%'   : '✓' if n_c/n >= MIN_ACCURACY else '✗',
        'RT mean (s)' : f'{rt.mean():.3f}' if len(rt) else 'n/a',
        'RT SD (s)'   : f'{rt.std():.3f}'  if len(rt) else 'n/a',
    })
tbl = pd.DataFrame(rows).set_index('Session')
print(f'Session Summary — {PARTICIPANT_ID}')
print(tbl.to_string())


---
## 3 Eye-Tracking Metrics: Focus on Saccade Amplitude & Latency

We focus on two key eye-tracking metrics to assess learning/practice effects:

- **Saccade latency** (ms): Time from stimulus onset to the first eye movement. This is pure **visual reaction time** — no motor component, just the brain's speed to detect and signal the eyes to move.
- **Saccade amplitude** (°): How far the eyes travel in that first saccade. This reflects **accuracy** — did the participant look at the correct location on the screen?

**Session 1 vs Session 2**: Comparing the first and second session reveals whether practice changes how quickly and accurately the eyes respond. Faster latency and more accurate amplitude suggest learning.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Load gaze metrics exported by VISION_longitudinal_gaze_metrics.ipynb
# ─────────────────────────────────────────────────────────────────

# Try longitudinal CSV first, then per-session CSVs
gaze_candidates = sorted(glob.glob(
    str(GAZE_METRICS_DIR / f'{PARTICIPANT_ID}_VISION_longitudinal_gaze_metrics.csv')
))
if not gaze_candidates:
    gaze_candidates = sorted(glob.glob(
        str(GAZE_METRICS_DIR / f'{PARTICIPANT_ID}_VISION_task_gaze_metrics*.csv')
    ))

if not gaze_candidates:
    print(f'⚠  No gaze metrics CSV found for {PARTICIPANT_ID} in {GAZE_METRICS_DIR}')
    print('   Run VISION_longitudinal_gaze_metrics.ipynb first to generate it.')
    all_tm = None
else:
    tm_parts = []
    for i, path in enumerate(gaze_candidates):
        tdf = pd.read_csv(path)
        if 'session' not in tdf.columns:
            tdf['session'] = f'S{i+1:03d}'
        tm_parts.append(tdf)
        print(f'  Loaded: {Path(path).name} — {len(tdf)} trials, sessions: {list(tdf["session"].unique())}')
    all_tm = pd.concat(tm_parts, ignore_index=True)
    sessions_sorted = sorted(all_tm['session'].unique())
    print(f'\nGaze metrics loaded: {len(all_tm)} trials across {len(sessions_sorted)} sessions: {sessions_sorted}')
    print(f'Saccade columns present: {[c for c in all_tm.columns if "sacc" in c]}')


In [ ]:
# ── PAIRED COMPARISON: Session 1 vs Session 2 ─────────────────────────────────
if all_tm is None:
    print('⚠  No gaze metrics loaded. Run the cell above first.')
else:
    sessions_sorted = sorted(all_tm['session'].unique())
    s1_lbl, s2_lbl = sessions_sorted[0], sessions_sorted[1]
    s1_data = all_tm[all_tm['session'] == s1_lbl]
    s2_data = all_tm[all_tm['session'] == s2_lbl]

    s1_latency   = s1_data['sacc_latency_ms'].dropna().values
    s2_latency   = s2_data['sacc_latency_ms'].dropna().values
    s1_amplitude = s1_data['sacc_amp_mean_deg'].dropna().values
    s2_amplitude = s2_data['sacc_amp_mean_deg'].dropna().values

    s1_lat_m, s1_lat_sem = s1_latency.mean(), s1_latency.std() / np.sqrt(len(s1_latency))
    s2_lat_m, s2_lat_sem = s2_latency.mean(), s2_latency.std() / np.sqrt(len(s2_latency))
    s1_amp_m, s1_amp_sem = s1_amplitude.mean(), s1_amplitude.std() / np.sqrt(len(s1_amplitude))
    s2_amp_m, s2_amp_sem = s2_amplitude.mean(), s2_amplitude.std() / np.sqrt(len(s2_amplitude))

    # (plotting code unchanged — runs with real data above)
    rng = np.random.default_rng(0)
    colors_sess = ['#1f77b4', '#ff7f0e']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (means, sems, vals_s1, vals_s2, ylabel, title) in zip(axes, [
        ([s1_lat_m, s2_lat_m], [s1_lat_sem, s2_lat_sem], s1_latency, s2_latency,
         'First Saccade Latency (ms)', 'Visual Reaction Time'),
        ([s1_amp_m, s2_amp_m], [s1_amp_sem, s2_amp_sem], s1_amplitude, s2_amplitude,
         'Mean Saccade Amplitude (°)', 'Eye Movement Accuracy'),
    ]):
        x_pos = [0, 1]
        ax.bar(x_pos, means, color=colors_sess, alpha=0.7, edgecolor='black', lw=1.5, width=0.5)
        ax.errorbar(x_pos, means, yerr=sems, fmt='none', color='black', capsize=10, lw=2)
        for i, (vals, x) in enumerate([(vals_s1, 0), (vals_s2, 1)]):
            ax.scatter(rng.normal(x, 0.04, len(vals)), vals, alpha=0.25, s=20, color=colors_sess[i])
        ax.set_xticks(x_pos); ax.set_xticklabels([s1_lbl, s2_lbl], fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=12); ax.set_title(title, fontsize=13, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')

    fig.suptitle(f'{PARTICIPANT_ID} — Saccade Metrics: {s1_lbl} vs {s2_lbl}', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    # ── TODO 4: Run a paired t-test comparing Session 1 vs Session 2 ─────────
    # Use scipy.stats.ttest_rel() on the two saccade latency arrays.
    # Print: t-statistic, p-value, and an interpretation sentence.
    # Do the same for saccade amplitude.

    n_paired = min(len(s1_latency), len(s2_latency))
    print('\n' + '='*70)
    print(f'PAIRED COMPARISON: {s1_lbl} vs {s2_lbl}')
    print('='*70)

    # Saccade latency:
    # t_lat, p_lat = ???
    # print(...)

    # Saccade amplitude:
    # t_amp, p_amp = ???
    # print(...)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Longitudinal trend: Saccade Latency and Amplitude across ALL sessions
# Uses real session-level aggregates from all_tm
# ─────────────────────────────────────────────────────────────────

if all_tm is None:
    print('⚠  No gaze metrics loaded. Run the loading cell first.')
else:
    # Compute mean ± SEM per session
    trend = (all_tm
             .groupby('session')[['sacc_latency_ms', 'sacc_amp_mean_deg']]
             .agg(['mean', 'sem'])
             .reset_index())
    trend.columns = ['session', 'lat_mean', 'lat_sem', 'amp_mean', 'amp_sem']
    trend = trend.sort_values('session').reset_index(drop=True)

    x = np.arange(len(trend))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── LEFT: Saccade Latency trend ──
    ax = axes[0]
    ax.plot(x, trend['lat_mean'].values, 'o-', color='steelblue', lw=2.5, ms=9, label='Mean latency')
    ax.fill_between(x,
                    trend['lat_mean'].values - trend['lat_sem'].values,
                    trend['lat_mean'].values + trend['lat_sem'].values,
                    alpha=0.2, color='steelblue', label='±SEM')
    ax.set_xticks(x)
    ax.set_xticklabels(trend['session'].values, fontsize=11, fontweight='bold')
    ax.set_ylabel('First Saccade Latency (ms)', fontsize=12, fontweight='bold')
    ax.set_title('Visual Reaction Time Trend', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=10)
    for xi, (m, s) in zip(x, zip(trend['lat_mean'].values, trend['lat_sem'].values)):
        ax.text(xi, m + s + 5, f'{m:.1f}', ha='center', fontsize=10, fontweight='bold')

    # ── RIGHT: Saccade Amplitude trend ──
    ax = axes[1]
    ax.plot(x, trend['amp_mean'].values, 's-', color='darkorange', lw=2.5, ms=9, label='Mean amplitude')
    ax.fill_between(x,
                    trend['amp_mean'].values - trend['amp_sem'].values,
                    trend['amp_mean'].values + trend['amp_sem'].values,
                    alpha=0.2, color='darkorange', label='±SEM')
    ax.set_xticks(x)
    ax.set_xticklabels(trend['session'].values, fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Saccade Amplitude (°)', fontsize=12, fontweight='bold')
    ax.set_title('Eye Movement Accuracy Trend', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=10)
    for xi, (m, s) in zip(x, zip(trend['amp_mean'].values, trend['amp_sem'].values)):
        ax.text(xi, m + s + 0.1, f'{m:.2f}°', ha='center', fontsize=10, fontweight='bold')

    fig.suptitle(f'{PARTICIPANT_ID} — Saccade Metrics Across All Sessions (mean ± SEM, real data)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    print('\nSession-level summary (real data):')
    print(trend.to_string(index=False))


---
## 6 Export & Summary

In [ ]:
# Export session-level saccade summary (real data)
if all_tm is None:
    print('⚠  No gaze metrics — nothing to export.')
else:
    sessions_sorted = sorted(all_tm['session'].unique())
    s1_lbl = sessions_sorted[0]
    s2_lbl = sessions_sorted[1] if len(sessions_sorted) > 1 else None

    rows_out = []
    for sess in sessions_sorted:
        d = all_tm[all_tm['session'] == sess]
        lat = d['sacc_latency_ms'].dropna()
        amp = d['sacc_amp_mean_deg'].dropna()
        rows_out.append({
            'session': sess,
            'n_trials': len(d),
            'sacc_latency_mean_ms': lat.mean(),
            'sacc_latency_sem_ms':  lat.sem(),
            'sacc_amp_mean_deg':    amp.mean(),
            'sacc_amp_sem_deg':     amp.sem(),
        })
    summary_export = pd.DataFrame(rows_out)

    out_path = GAZE_METRICS_DIR / f'{PARTICIPANT_ID}_saccade_longitudinal_summary.csv'
    summary_export.to_csv(out_path, index=False)
    print(f'Saved: {out_path}')
    print('\n' + summary_export.round(3).to_string(index=False))

    # Print improvement if 2 sessions
    if s2_lbl and len(sessions_sorted) >= 2:
        r1 = summary_export[summary_export['session'] == s1_lbl].iloc[0]
        r2 = summary_export[summary_export['session'] == s2_lbl].iloc[0]
        delta_lat = r1['sacc_latency_mean_ms'] - r2['sacc_latency_mean_ms']
        delta_amp = r2['sacc_amp_mean_deg']    - r1['sacc_amp_mean_deg']
        print(f'\nChange {s1_lbl} → {s2_lbl}:')
        print(f'  Latency:   {delta_lat:+.1f} ms  ({"faster" if delta_lat > 0 else "slower"})')
        print(f'  Amplitude: {delta_amp:+.2f}°   ({"larger" if delta_amp > 0 else "smaller"})')
